In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

bike_sharing_demand_path = kagglehub.competition_download('bike-sharing-demand')

print('Data source import complete.')


# 데이터 불러오기

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
data_path = '/kaggle/input/bike-sharing-demand/'

In [ ]:
train = pd.read_csv(data_path + 'train.csv')
test = pd.read_csv(data_path + 'test.csv')
submission = pd.read_csv(data_path + 'sampleSubmission.csv')

In [ ]:
train.shape, test.shape

# 피처 엔지니어링

In [ ]:
train = train[train['weather'] != 4]

In [ ]:
all_data_temp = pd.concat([train, test])
all_data_temp

In [ ]:
all_data = pd.concat([train, test], ignore_index=True)
all_data

In [ ]:
from datetime import datetime

In [ ]:
# all_data['date'] = all_data['datetime'].apply(lambda x: x.split()[0])
# all_data['year'] = all_data['datetime'].apply(lambda x: x.split()[0].split('-')[0])
# all_data['month'] = all_data['datetime'].apply(lambda x: x.split()[0].split('-')[1])
# all_data['hour'] = all_data['datetime'].apply(lambda x: x.split()[1].split(':')[0])
# all_data['weekday'] = all_data['date'].apply(lambda dateString: datetime.strptime(dateString, "%Y-%m-%d").weekday())

all_data['datetime'] = pd.to_datetime(all_data['datetime'])
all_data['year'] = all_data['datetime'].dt.year
all_data['month'] = all_data['datetime'].dt.month
all_data['hour'] = all_data['datetime'].dt.hour
all_data['weekday'] = all_data['datetime'].dt.weekday

In [ ]:
drop_features = ['casual', 'registered', 'datetime', 'month', 'windspeed'] # , 'date'
all_data = all_data.drop(drop_features, axis=1)

In [ ]:
X_train = all_data[~pd.isnull(all_data['count'])]
X_test = all_data[pd.isnull(all_data['count'])]

In [ ]:
X_train = X_train.drop(['count'], axis=1)
X_test = X_test.drop(['count'], axis=1)

In [ ]:
y = train['count']

In [ ]:
X_train.head()

# 평가지표 계산 함수 작성

In [ ]:
def rmsle(y_true, y_pred, convertExp=True):
    if convertExp:
        y_true = np.exp(y_true)
        y_pred = np.exp(y_pred)

    log_true = np.nan_to_num(np.log(y_true+1))
    log_pred = np.nan_to_num(np.log(y_pred+1))

    output = np.sqrt(np.mean((log_true - log_pred)**2))
    return output

# 모델 훈련

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
linear_reg_model = LinearRegression()

In [ ]:
log_y = np.log(y)
linear_reg_model.fit(X_train, log_y)

# 모델 성능 검증

In [ ]:
preds = linear_reg_model.predict(X_train)

In [ ]:
print(f'선형 회귀의 RMSLE 값 : {rmsle(log_y, preds, True):.4f}')

# 예측 및 결과 제출

In [ ]:
linearreg_preds = linear_reg_model.predict(X_test)

In [ ]:
submission['count'] = np.exp(linearreg_preds)
submission.to_csv('submission.csv', index=False)